In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# A Volume is Unity Catalog's place for raw files (think: a cloud storage folder)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_landing")

In [0]:
import random, os
import pandas as pd
from datetime import datetime, timedelta

random.seed(99)   # different seed, otherwise you'd get identical data
base = "/Volumes/workspace/bronze/raw_landing"

statuses = ["completed", "completed", "completed", "shipped", "cancelled", "returned"]
orders = []
for i in range(20001, 25001):   # new order_ids, continuing after batch 1
    status = random.choice(statuses)
    if random.random() < 0.2: status = status.upper()
    qty = random.randint(1, 5)
    if random.random() < 0.01: qty = -qty
    orders.append({
        "order_id": i,
        "customer_id": random.randint(1, 500),
        "product_id": random.randint(1, 50),
        "quantity": qty,
        "order_ts": (datetime(2026, 1, 1) + timedelta(minutes=random.randint(0, 300000))).strftime("%Y-%m-%d %H:%M:%S"),
        "status": status,
    })
orders += random.sample(orders, 50)   # some duplicates within the batch

pd.DataFrame(orders).to_csv(f"{base}/orders/orders_batch2.csv", index=False)
print(os.listdir(f"{base}/orders"))   # should now list batch1 AND batch2

In [0]:
import random, os
import pandas as pd
from datetime import datetime, timedelta

random.seed(123)   # new seed, so the data differs from batch 2
base = "/Volumes/workspace/bronze/raw_landing"

statuses = ["completed", "completed", "completed", "shipped", "cancelled", "returned"]
orders = []
for i in range(25001, 26001):   # 1,000 brand-new order_ids
    status = random.choice(statuses)
    if random.random() < 0.2: status = status.upper()
    qty = random.randint(1, 5)
    if random.random() < 0.01: qty = -qty
    orders.append({
        "order_id": i,
        "customer_id": random.randint(1, 500),
        "product_id": random.randint(1, 50),
        "quantity": qty,
        "order_ts": (datetime(2026, 3, 1) + timedelta(minutes=random.randint(0, 100000))).strftime("%Y-%m-%d %H:%M:%S"),
        "status": status,
    })
orders += random.sample(orders, 20)   # 20 duplicates inside the batch

pd.DataFrame(orders).to_csv(f"{base}/orders/orders_batch3.csv", index=False)
print(os.listdir(f"{base}/orders"))

In [0]:
import json, random, os
from datetime import datetime, timedelta

random.seed(7)
base = "/Volumes/workspace/bronze/raw_landing/events"
os.makedirs(base, exist_ok=True)

event_types = ["page_view", "page_view", "page_view", "add_to_cart", "purchase"]
events = []
for i in range(30000):
    ev = {
        "event_id": f"e{i}",
        "customer_id": random.randint(1, 500),
        "event_type": random.choice(event_types),
        "event_ts": (datetime(2025, 1, 1) + timedelta(seconds=random.randint(0, 5000000))).isoformat(),
        "device": {"type": random.choice(["mobile", "desktop", "tablet"]),
                   "os": random.choice(["ios", "android", "windows", "macos"])},
        "page": {"url": f"/product/{random.randint(1, 50)}",
                 "referrer": random.choice(["google", "email", "direct", None])},
    }
    if random.random() < 0.02: ev["event_type"] = ev["event_type"].upper()   # messy casing
    if random.random() < 0.01: ev["event_type"] = "unknown_event"             # invalid value
    if random.random() < 0.01: ev["customer_id"] = None                       # missing customer
    events.append(ev)

events += random.sample(events, 300)   # duplicates

with open(f"{base}/events_batch1.json", "w") as f:
    f.write("\n".join(json.dumps(e) for e in events))

print(os.listdir(base))

In [0]:
import pandas as pd

base = "/Volumes/workspace/bronze/raw_landing"
df = pd.read_csv(f"{base}/products/products_batch1.csv")

names = {
    "Electronics": ["Wireless Earbuds", "Bluetooth Speaker", "Smart Watch", "USB-C Hub", "Webcam", "Power Bank", "Laptop Stand", "Gaming Mouse", "Mechanical Keyboard", "Tablet", "Monitor", "Phone Charger"],
    "Clothing": ["Cotton T-Shirt", "Denim Jeans", "Hoodie", "Running Shoes", "Winter Jacket", "Wool Socks", "Baseball Cap", "Rain Coat", "Polo Shirt", "Leggings", "Scarf", "Sweater"],
    "Home": ["Desk Lamp", "Coffee Maker", "Throw Blanket", "Candle Set", "Storage Box", "Wall Clock", "Cushion Cover", "Plant Pot", "Towel Set", "Vacuum Flask", "Cutting Board", "Photo Frame"],
    "Books": ["Mystery Novel", "Cookbook", "Travel Guide", "Biography", "Science Fiction", "History Book", "Self-Help Guide", "Children's Story", "Poetry Collection", "Graphic Novel", "Atlas", "Journal"],
}

counters = {}
new_names = []
for cat in df["category"]:
    i = counters.get(cat, 0)
    new_names.append(f"{names[cat][i % len(names[cat])]} {i // len(names[cat]) + 1}" if i >= len(names[cat]) else names[cat][i])
    counters[cat] = i + 1

df["product_name"] = new_names
df.to_csv(f"{base}/products/products_batch1.csv", index=False)
print(df[["product_id", "product_name", "category"]].head(10))